# Laboratorio 2 · Bitácora

**Nombre:**  
**Usuario de GitHub:**  
**Fecha:**  

---

> Los enunciados están en la guía del laboratorio. Aquí solo van tus
> predicciones, tus resultados y tus explicaciones.

> **La regla:** la predicción se escribe ANTES de ejecutar la celda de código
> que tiene debajo. Equivocarse no resta. Rellenarla después, sí.

> **Lo nuevo de hoy:** los métodos de esta sesión son aleatorios. Una sola
> ejecución no es una medición. A partir del ejercicio 2, todo número que
> escribas aquí tiene que venir con su intervalo y con cuántas semillas lo
> produjeron.


## Preparación


In [1]:
import numpy as np

from rlrs.dp import value_iteration
from rlrs.envs import ARROWS, GridWorld, acantilado
from rlrs.evaluation import evaluate
from rlrs.policies import EpsilonAvidaPolicy, GreedyTabularPolicy
from rlrs.td import error_frente_a, mc_control, q_learning, sarsa

# Si esta celda falla, para y resuélvelo antes de seguir.
print('todo importado')


todo importado


## Mi variante

La misma de ayer. Si no la anotaste, ejecuta `uv run python scripts/variante.py`.

> Obtenido de ejecutar script variante que entrega los valores para el laboratorio
```
  Variante de  charliemedinar
    ruido           0.2
    coste por paso  -0.1
    gamma           0.9   (igual para todos)
```


In [2]:
RUIDO = 0.2   # <- rellena, el mismo de ayer
COSTE = -0.1   # <- rellena, el mismo de ayer
GAMMA = 0.9

mi_env = GridWorld(noise=RUIDO, step_reward=COSTE)

# La respuesta conocida: tu V* de ayer. Es contra esto que medimos hoy.
optimos, politica_optima, barridos = value_iteration(mi_env, gamma=GAMMA)
print(f'{barridos} barridos'); print(mi_env.render_values(optimos, politica_optima))


35 barridos
+0.26>  +0.45>  +0.65>  +0.91>   +1    
+0.10^    ###   +0.46^  +0.50^   -1    
-0.02^  +0.09>  +0.26^    ###   -0.28v 
-0.13^  -0.04^  +0.08^  -0.05<  -0.18< 


### Dos ayudas que se usan en todo el cuaderno


In [3]:
libres = [(r, c) for r in range(mi_env.n_rows) for c in range(mi_env.n_cols)
          if not mi_env.is_wall((r, c)) and not mi_env.is_terminal((r, c))]


def coincidencias(q):
    """En cuántas casillas la acción ávida de q es la acción óptima."""
    return sum(int(q[mi_env.state_index(p)].argmax()
                   == politica_optima[mi_env.state_index(p)]) for p in libres)


def intervalo(xs):
    """Media e intervalo de confianza al 95 %. Devuelve (media, bajo, alto)."""
    a = np.array(xs, dtype=float)
    media = a.mean()
    mitad = 1.96 * a.std(ddof=1) / np.sqrt(len(a)) if len(a) > 1 else 0.0
    return media, media - mitad, media + mitad


print(f'{len(libres)} casillas libres')


16 casillas libres


---
## Ejercicio 1 · Los tres métodos contra la respuesta conocida


**Antes de ejecutar.** Ordena los tres métodos de menor a mayor error, y di por qué crees que ese es el orden.

_Tu predicción:_  
> Mi prediccion de que metodos van de menor a mayor error son:
>  - Q-Learning
>  - SARSA
>  - MonteCarlo

> Mi prediccion se basa a traves del estudio, que nos da a entender que Q-Learning esta enfocada en optimizar a medida que avanza y encuentra mejor la ruta optima que los otros, y MonteCarlo es el que mayor error puede generar porque su estrategia es basica y sin mucho control ademas que para obtener resultados tiene que terminar el recorrido para saber si le fue bien o no



In [4]:
for nombre, metodo in (('monte-carlo', mc_control), ('sarsa', sarsa), ('q-learning', q_learning)):
    ap = metodo(mi_env, episodes=5000, gamma=GAMMA, seed=0)
    err = error_frente_a(ap.q, optimos, mi_env)
    print(f'{nombre:<12} error {err:.4f}   política {coincidencias(ap.q)}/{len(libres)}')


monte-carlo  error 0.5158   política 14/16
sarsa        error 0.2489   política 14/16
q-learning   error 0.1959   política 13/16


**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_

> El orden del error sí coincidió con lo que predije: Q-learning quedó con el error más bajo (0.1959), después SARSA (0.2489) y por último Monte Carlo (0.5158), tal como pensaba, porque Q-learning optimiza mejor la ruta y Monte Carlo tiene más variación al depender de que termine todo el episodio.

> Lo que no predije, y me pareció curioso, es que el conteo de política no siguió el mismo orden: Monte Carlo acertó 14/16, SARSA también 14/16, y Q-learning, el de menor error, solo acertó 13/16. El que mejor estimó los valores fue el que menos veces acertó la acción correcta.

> Esto pasa porque error y política miden cosas distintas: el error mide qué tan cerca está el VALOR numérico de la mejor acción respecto al real, no si la acción elegida es la correcta. En una casilla como (3,1), donde las dos mejores acciones están casi empatadas (diferencia de 0.0001), cualquier método puede fallar en elegir la "correcta" sin que eso le cueste casi nada de error, porque las dos opciones valen casi lo mismo.

> También noté que SARSA y Q-learning fallaron exactamente en las mismas dos casillas, (1,3) y (2,4), pegadas a la trampa, mientras que Monte Carlo no falló ahí. Esto no es casualidad: los dos son métodos de diferencias temporales, que se apoyan en su propia estimación (aún no confirmada) para corregir en cada paso, y ese apoyo puede arrastrar un sesgo sistemático cerca de estados riesgosos. Monte Carlo, al esperar siempre el resultado real del episodio completo, no tiene ese sesgo, pero paga el precio con más varianza en otras casillas, lo que explica su error total más alto.


---
## Ejercicio 2 · Un número sin intervalo, otra vez

> Esta celda tarda cerca de medio minuto. No se colgó.


**Antes de ejecutar.** ¿Se va a mantener el orden del ejercicio 1 con cinco semillas? ¿Y van las dos cifras, el error y el recuento de política, a contar la misma historia?

_Tu predicción:_

> El orden de qué método tiene menor a mayor error se va a mantener, pienso que la aleatoriedad no va a influir sobre eso por la misma lógica de cada uno; las cifras sí pueden diferir un poco, porque los valores sobre los que se evalúa en los pasos y demás ahí sí son distintos.

> El conteo de política va a ser similar también, no creo que vaya a tener mucha diferencia del uno al otro.


In [6]:
for nombre, metodo in (('monte-carlo', mc_control), ('sarsa', sarsa), ('q-learning', q_learning)):
    errores, politicas = [], []
    for semilla in range(5):
        ap = metodo(mi_env, episodes=5000, gamma=GAMMA, seed=semilla)
        errores.append(error_frente_a(ap.q, optimos, mi_env))
        politicas.append(coincidencias(ap.q))
    e, elo, ehi = intervalo(errores)
    p, plo, phi = intervalo(politicas)
    print(f'{nombre:<12} error {e:.4f} [{elo:.4f}, {ehi:.4f}]'
          f'   política {p:.1f} [{plo:.1f}, {phi:.1f}]')


monte-carlo  error 0.5238 [0.4918, 0.5558]   política 12.6 [10.8, 14.4]
sarsa        error 0.2359 [0.1807, 0.2912]   política 13.8 [13.1, 14.5]
q-learning   error 0.2169 [0.1969, 0.2369]   política 13.2 [12.5, 13.9]


**Con los intervalos delante, responde las dos por separado.**

1. ¿El **error** distingue a los tres métodos, o hay parejas cuyos intervalos se solapan?

   _Tu respuesta:_ Distingue parcialmente. Monte Carlo queda claramente separado de los otros dos: su intervalo [0.4918, 0.5558] no se toca con el de SARSA [0.1807, 0.2912] ni con el de Q-learning [0.1969, 0.2369], así que ahí sí hay evidencia real de que Monte Carlo tiene más error. Pero SARSA y Q-learning SÍ se solapan (el intervalo de Q-learning cabe casi entero dentro del de SARSA), así que con estas 5 semillas no se puede afirmar con confianza que uno le gane al otro en error, aunque el promedio de Q-learning (0.2169) sea un poco más bajo que el de SARSA (0.2359).

2. ¿El **recuento de política** los distingue?

   _Tu respuesta:_ No. Los tres intervalos se solapan entre sí: Monte Carlo [10.8, 14.4], SARSA [13.1, 14.5] y Q-learning [12.5, 13.9] se tocan por parejas (MC con SARSA, MC con Q-learning, y SARSA con Q-learning). Con 5 semillas no hay evidencia suficiente para decir que alguno de los tres acierta más casillas de política que los otros, aunque el promedio de Monte Carlo (12.6) parezca más bajo a simple vista.



**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_ En parte coincidió. El orden de los promedios de error sí se mantuvo (Q-learning, SARSA, Monte Carlo, de menor a mayor), pero no había considerado que ese orden no es igual de sólido entre los tres: Monte Carlo sí queda claramente separado de los otros dos, pero SARSA y Q-learning se solapan entre sí. Con estas 5 semillas puedo afirmar que Monte Carlo tiene más error que los otros dos, pero no puedo afirmar que Q-learning sea mejor que SARSA en error.

Sobre la política, mi predicción de que iba a ser similar entre los tres sí se cumplió, aunque no de la forma que pensé: los tres intervalos se solapan entre sí, no solo SARSA y Q-learning como noté al principio, sino también Monte Carlo con los otros dos. Con esto entendí que el error y la política cuentan historias distintas: el error separa con claridad a Monte Carlo del resto, pero ninguna de las dos cifras separa con confianza a SARSA de Q-learning.


---
## Ejercicio 3 · Apagar la exploración


**Antes de ejecutar.** Con $\varepsilon = 0$ el agente siempre toma la acción que ahora mismo cree mejor. ¿Aprenderá la política óptima, una peor, o depende de la suerte inicial? Y con $\varepsilon = 0{,}5$: ¿mejor o peor que con $0{,}1$?

_Tu predicción:_

> Mi predicción al respecto es que cuando ε es 0 quiere decir que la aleatoriedad es nula, no va a haber ese lanzado de dado para ver hacia dónde voy a intentar, sino que con base a la suerte inicial va a partir siempre, sin importar las semillas e iteraciones.

> Con 0,5 es una aleatoriedad balanceada, lo que puede causar que en un inicio, por encontrar el camino, vaya bien, pero con el paso de movimientos e iteraciones pierda precisión en la política.

> Con 0,1 es una aleatoriedad inicial buena, pero lo determinístico se va a imponer finalmente.


In [7]:
for eps in (0.0, 0.05, 0.1, 0.3, 0.5):
    errores, politicas = [], []
    for semilla in range(5):
        ap = sarsa(mi_env, episodes=5000, gamma=GAMMA,
                   epsilon=eps, epsilon_final=eps, seed=semilla)   # sin decaimiento
        errores.append(error_frente_a(ap.q, optimos, mi_env))
        politicas.append(coincidencias(ap.q))
    e, elo, ehi = intervalo(errores)
    p, plo, phi = intervalo(politicas)
    print(f'eps {eps:<5} error {e:.4f} [{elo:.4f}, {ehi:.4f}]'
          f'   política {p:.1f} [{plo:.1f}, {phi:.1f}]')


eps 0.0   error 0.1410 [0.0902, 0.1918]   política 13.0 [12.4, 13.6]
eps 0.05  error 0.1972 [0.1491, 0.2452]   política 13.8 [13.1, 14.5]
eps 0.1   error 0.2575 [0.1861, 0.3289]   política 14.4 [13.9, 14.9]
eps 0.3   error 0.3364 [0.2634, 0.4094]   política 14.6 [13.8, 15.4]
eps 0.5   error 0.5259 [0.5112, 0.5406]   política 15.0 [14.4, 15.6]


**Son dos fallos distintos.** Nombra por separado qué le falta al agente de $\varepsilon = 0$ y qué le sobra al de $\varepsilon = 0{,}5$.

_Tu respuesta:_

> A ε=0 le falta cobertura. Como nunca prueba nada distinto de lo que ya cree mejor, si por la semilla nunca visita cierto par (casilla, acción), esa entrada de la tabla Q se queda en el valor con el que arrancó (cero) y nunca se corrige. Ahí la "elección" no es una decisión informada, es lo que quedó por defecto — eso es lo que le castiga la política (13.0, la más baja de las cinco). En las casillas que sí visitó lo suficiente, en cambio, aprendió el valor real sin el ruido de comportarse al azar, por eso su error es el más bajo de los cinco (0.1410).

> A ε=0.5 le sobra comportamiento aleatorio metido dentro de lo que aprende. SARSA es un método dentro de política: no aprende "el mejor camino posible", aprende "cuánto vale comportarme tal cual me estoy comportando ahora". Con ε=0.5 se mueve al azar la mitad del tiempo, un comportamiento bastante malo, y el valor que aprende (Q) refleja el valor de ESE comportamiento, lejos del óptimo real — por eso su error es el más alto de los cinco (0.5259). Lo que sí gana con tanta exploración es cobertura total: visita todas las combinaciones estado-acción muchas veces, así que ninguna casilla se queda "en blanco" como le pasa a ε=0, y por eso su política termina siendo, aunque parezca contradictorio, la más precisa de las cinco (15.0).


**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_

> Definitivamente estaba mal con mi predicción. Esperaba que ε=0 aprendiera la mejor política porque al no haber aleatoriedad pensé que iba directo a lo determinístico, y que ε=0.5 saliera peor en todo por ser puro azar. Encontré justo lo contrario: ε=0 tiene el error más bajo pero la política más floja (13.0/16), y ε=0.5 tiene el error más alto pero la política más precisa (15.0/16).

> Lo que no había considerado es que error y política no miden lo mismo, y que cada ε falla por una razón distinta. A ε=0 le falta cobertura: partes de la tabla nunca se visitan y se quedan sin aprender nada, lo que arruina la política ahí. A ε=0.5 le sobra comportamiento aleatorio metido en lo que aprende, porque SARSA aprende el valor de cómo se comporta de verdad (no el de la política óptima), y comportarse la mitad del tiempo al azar aleja mucho ese valor del real — pero esa misma exploración constante le da cobertura completa, y por eso acierta más veces la acción correcta aunque sus números estén más lejos del valor verdadero. Con esto entendí que las semillas, el ruido y epsilon sí son factores importantes, pero no todos actúan en la misma dirección ni sobre la misma cifra.


---
## Ejercicio 4 · El tamaño del paso


**Antes de ejecutar.** ¿El error va a bajar monótonamente al subir $\alpha$, va a subir monótonamente, o va a tener un mínimo en algún punto intermedio? Apuesta por una de las tres formas.

_Tu predicción:_

> Alfa impacta en la forma en que aprende a nivel de Q, qué tan rápido. Con alfa bajo, va a necesitar muchas más iteraciones; finalmente resolverá el camino, pero el rate de política se puede ver impactado.

> Si alfa es alto, va a ser más rápido que su contraparte, pero puede que no aprenda del todo bien y su error sea más alejado de lo óptimo.

> Mi apuesta final por una de las tres formas: el error va a subir a medida que alfa aumente (monótono creciente), no un mínimo en un punto intermedio.


In [9]:
for alpha in (0.01, 0.1, 0.5, 0.9):
    errores = [error_frente_a(sarsa(mi_env, episodes=5000, gamma=GAMMA,
                                    alpha=alpha, seed=s).q, optimos, mi_env)
               for s in range(5)]
    e, elo, ehi = intervalo(errores)
    print(f'alpha {alpha:<5} error {e:.4f} [{elo:.4f}, {ehi:.4f}]')


alpha 0.01  error 0.2518 [0.2084, 0.2952]
alpha 0.1   error 0.2359 [0.1807, 0.2912]
alpha 0.5   error 0.4566 [0.4122, 0.5009]
alpha 0.9   error 1.1386 [0.9066, 1.3706]


**Distingue los dos problemas.** El de $\alpha$ muy pequeño y el de $\alpha$ muy grande no son el mismo.

_Tu respuesta:_

> Un alfa demasiado pequeño causa que el aprendizaje sea extremadamente lento, o que se aprenda muy poco por episodio — por lo que no sabemos con certeza si α=0.01 avanzó más en el aprendizaje que α=0.1: sus intervalos, [0.2084, 0.2952] y [0.1807, 0.2912], se solapan casi por completo, así que con estas 5 semillas esa diferencia podría ser puro ruido.

> Un alfa muy alto genera inestabilidad. En la regla de actualización, `Q(s,a) ← Q(s,a) + α·δ`, con α cerca de 1 cada actualización casi reemplaza por completo la estimación acumulada con el dato de ese solo paso — y ese dato trae el ruido propio del entorno (mi variante tiene noise=0.2). Entonces la tabla Q no promedia el ruido con el tiempo, lo persigue: salta de un valor a otro cada vez que le toca una transición distinta, y nunca se asienta cerca de un valor estable. De ahí el salto a error 1.1386 en α=0.9, muy por encima de los demás.


**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_

> Coincidió a medias. Apostaba por "monótono creciente" en los cuatro puntos, y de α=0.1 en adelante eso se cumplió con fuerza: 0.2359 → 0.4566 → 1.1386, con intervalos que no se tocan entre sí, así que ese tramo es un ascenso real, no ruido.

> Lo que no esperaba es lo que pasó entre α=0.01 y α=0.1: el promedio en realidad bajó un poco (0.2518 → 0.2359) en vez de subir, aunque sus intervalos, [0.2084, 0.2952] y [0.1807, 0.2912], se solapan casi por completo — con estas 5 semillas no puedo afirmar que haya una diferencia real ahí, podría ser puro ruido. Entonces mi predicción de "monótono creciente" no es del todo exacta: es más bien plano/empatado entre 0.01 y 0.1, y después sube con claridad.

> Esto tiene sentido con los dos problemas que ya había distinguido: a α=0.01 le falta tiempo para aprender (por eso no bajó su error respecto a 0.1, no alcanzó a asentarse mejor en los mismos 5000 episodios), y a α alto le sobra inestabilidad (por eso el ascenso desde 0.1 es tan marcado). Son dos mecanismos distintos, y por eso la forma real no es una línea perfectamente recta creciente, aunque la tendencia general sí va para arriba.


---
## Ejercicio 5 · El error plantado

Este no lleva código propio. Ejecuta en la terminal:

```
uv run python experiments/sin_modelo.py --parte 3
```


**Antes de ejecutar.** ¿Cuál de las dos formas de medir va a dar un retorno peor, y por qué? ¿Y cuánto peor, un poco o mucho?

_Tu predicción:_ No alcancé a formular una predicción propia antes de ejecutar — la distinción entre entrenar con ε-ávida y medir con ávida todavía no me quedaba clara en ese momento, así que preferí no adivinar a ciegas.



**Pega aquí la salida del guion.**

```
UPTC · Sesion 2 · Aprender sin modelo del entorno

  3 · El error plantado: medir con la politica que exploraba

  como se mide             retorno                IC 95%   exito
  --------------------------------------------------------------
  avida (epsilon = 0)       +0.654  [+0.640, +0.669]  100.0%
  epsilon = 0.05            +0.646  [+0.631, +0.660]  100.0%
  epsilon = 0.1             +0.617  [+0.596, +0.638]   99.7%
  epsilon = 0.3             +0.429  [+0.380, +0.478]   96.7%

  Es el MISMO agente en las cuatro filas. Lo unico que cambia es si
  sigue explorando mientras se le mide. Con epsilon = 0.3 los
  intervalos ni siquiera se solapan con los de epsilon = 0: la
  conclusion equivocada seria estadisticamente significativa.
```



**El diagnóstico.** ¿Por qué esa medición está mal hecha, y qué está midiendo en realidad? Y en una frase: ¿cuándo sí tendría sentido medir con la política que explora?

_Tu respuesta:_

> Es el mismo agente, con la misma tabla Q, en las cuatro filas — nada de eso cambió. Lo único que cambió fue cuánto se le dejó seguir tirando el dado mientras se le medía. Por eso la medición con ε=0.3 está mal hecha: cada vez que el dado le dice explorar, el agente ignora a propósito lo que ya sabe que es la mejor acción y hace algo al azar en su lugar, y eso tira a la basura una parte de la recompensa que sí sabía cómo conseguir. La caída de +0.654 a +0.429 no dice que el agente sepa menos; dice que se le impidió usar, una fracción del tiempo, lo que ya había aprendido.

> Lo que esa medición mide en realidad no es "qué tan bueno es el agente", sino "qué tan bien rinde una mezcla de buenas decisiones (la mayoría) con decisiones al azar (una fracción ε)". Es una mezcla, no una medida limpia del conocimiento aprendido.

> Medir con la política que explora sí tendría sentido cuando el sistema, en producción real, nunca va a dejar de explorar — por ejemplo, un recomendador que sigue probando cosas nuevas con usuarios reales de forma indefinida. Ahí, el número con exploración es el honesto, porque así es exactamente como se va a comportar en la práctica.


---
## Ejercicio 6 · El acantilado


**Antes de ejecutar.** ¿Cuál de los dos métodos va a ganar? Escríbelo, y después vuelve a leer la pregunta: ¿tiene sentido tal como está formulada?

_Tu predicción:_

> Depende de lo que estemos buscando. Si buscamos quién llega más rápido a la meta (en menor cantidad de pasos), Q-learning va a destacar, aunque medirlo de manera aleatoria (con exploración) pueda causarle más posibilidad de error. En el caso contrario, SARSA va a tardar más en llegar a la meta, pero es menos propenso al error, y las mediciones lo van a demostrar.

> La pregunta "¿cuál de los dos métodos va a ganar?" no tiene una sola respuesta correcta — depende de si el sistema se va a seguir explorando después de entrenar o no.


In [10]:
cl = acantilado()

for nombre, metodo in (('sarsa', sarsa), ('q-learning', q_learning)):
    avido, explorando, entrenamiento = [], [], []
    for semilla in range(5):
        ap = metodo(cl, episodes=5000, gamma=1.0, alpha=0.1, seed=semilla)
        ev = evaluate(acantilado(), GreedyTabularPolicy(ap.q.argmax(axis=1)),
                      episodes=100, base_seed=0)
        avido.append(ev.mean)
        ex = evaluate(acantilado(), EpsilonAvidaPolicy(ap.q, 0.05),
                      episodes=100, base_seed=0)
        explorando.append(ex.mean)
        entrenamiento.append(float(np.mean(ap.retornos[-500:])))
    a, alo, ahi = intervalo(avido)
    x, xlo, xhi = intervalo(explorando)
    t, tlo, thi = intervalo(entrenamiento)
    print(f'{nombre:<11} ávido {a:+.2f} [{alo:+.2f}, {ahi:+.2f}]'
          f'   explorando {x:+.2f} [{xlo:+.2f}, {xhi:+.2f}]'
          f'   entrenamiento {t:+.2f} [{tlo:+.2f}, {thi:+.2f}]')


sarsa       ávido -16.00 [-16.00, -16.00]   explorando -16.98 [-17.01, -16.96]   entrenamiento -18.24 [-18.72, -17.75]
q-learning  ávido -12.00 [-12.00, -12.00]   explorando -24.86 [-25.92, -23.80]   entrenamiento -27.01 [-28.37, -25.66]


### Los dos caminos


In [11]:
for nombre, metodo in (('sarsa', sarsa), ('q-learning', q_learning)):
    ap = metodo(cl, episodes=5000, gamma=1.0, alpha=0.1, seed=0)
    print(f'\n{nombre}:')
    print(cl.render_values(ap.q.max(axis=1), ap.q.argmax(axis=1)))



sarsa:
-14.13>  -12.99>  -11.87>  -10.66>  -9.57>  -8.53>  -7.58>  -6.80>  -5.52>  -4.45>  -3.36>  -2.22v 
-15.37^  -14.40^  -13.58^  -12.47^  -11.95>  -10.16>  -8.90>  -7.71^  -4.53>  -3.41>  -2.44>  -1.02v 
-16.56^  -15.95^  -16.56^  -15.67^  -13.24^  -12.32^  -11.35^  -9.09>  -7.22^  -5.72^  -1.35>  +0.00v 
-17.67^   -100      -100      -100      -100      -100      -100      -100      -100      -100      -100      +0    

q-learning:
-11.60^  -11.03^  -10.32^  -9.52>  -8.66>  -7.79>  -6.87>  -5.92>  -4.96>  -3.98>  -2.99>  -2.00v 
-12.00>  -11.00>  -10.00>  -9.00v  -8.00>  -7.00>  -6.00v  -5.00>  -4.00>  -3.00v  -2.00v  -1.00v 
-11.00>  -10.00>  -9.00>  -8.00>  -7.00>  -6.00>  -5.00>  -4.00>  -3.00>  -2.00>  -1.00>  +0.00v 
-12.00^   -100      -100      -100      -100      -100      -100      -100      -100      -100      -100      +0    


**La explicación del mecanismo.** Escribe las dos reglas de actualización una debajo de la otra y subraya lo único que cambia: qué valor se usa para el estado siguiente. Desde ahí, explica por qué cada método aprende el camino que aprende.

_Tu respuesta:_

> SARSA:      `Q(s,a) ← Q(s,a) + α[r + γ·Q(s',a') − Q(s,a)]`
>
> Q-learning: `Q(s,a) ← Q(s,a) + α[r + γ·max_a' Q(s',a') − Q(s,a)]`
>
> Lo único que cambia es qué valor representa "el estado siguiente": SARSA usa `Q(s',a')`, el valor de la acción `a'` que el agente **de verdad va a ejecutar** (elegida con la misma política ε-ávida, exploración incluida). Q-learning usa `max_a' Q(s',a')`, el valor de la **mejor** acción posible en `s'`, la vaya a tomar o no.

> Esa única diferencia explica los dos caminos. Cerca del borde del acantilado, cuando el agente explora (tira el dado), a veces el paso que de verdad ejecuta lo tira al precipicio (−100). SARSA usa exactamente esa acción real, con sus tropiezos incluidos, para actualizar — así que las casillas junto al borde terminan con un valor castigado por esos tropiezos que sí ocurrieron durante el entrenamiento, y el agente aprende a alejarse. Q-learning, en cambio, siempre actualiza asumiendo que a partir del estado siguiente se juega perfecto (el máximo), sin importar qué acción se ejecutó realmente — así que un tropiezo ocasional durante el entrenamiento no le "ensucia" el valor de las casillas junto al borde, y esas casillas mantienen un valor optimista que sí refleja el camino más corto. Por eso SARSA rodea por la fila de arriba y Q-learning camina pegado al precipicio.


**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_

> Coincidió con bastante precisión. Predije que Q-learning destacaría en velocidad y que la medición aleatoria le causaría más error, y eso pasó exactamente: ávido, Q-learning llega en 12 pasos (−12.00) contra los 16 de SARSA (−16.00), pero al medir explorando (ε=0.05) Q-learning se desploma a −24.86 mientras SARSA apenas empeora a −16.98.

> También predije que SARSA tardaría más pero sería menos propenso al error, y las mediciones lo confirmaron: su intervalo ávido a explorando casi no se mueve, mientras que el de Q-learning se abre mucho y empeora fuerte.

> Y mi respuesta a la segunda parte —que la pregunta "¿cuál gana?" no tiene una sola respuesta, depende de si el sistema va a seguir explorando— también se sostuvo: bajo la medición ávida gana Q-learning, y bajo la medición con exploración (o mirando el propio retorno de entrenamiento) gana SARSA por bastante margen. Los dos caminos dibujados lo confirman: SARSA sube a la fila más alejada del precipicio y solo baja al final, mientras Q-learning camina toda la ruta pegado al borde, una fila arriba del acantilado.


---
## Antes de entregar

- [ ] Las seis predicciones están escritas, y se escribieron antes de ejecutar.
- [ ] Todos los números llevan su intervalo y dicen cuántas semillas los produjeron.
- [ ] Ninguna conclusión dice más de lo que los intervalos permiten decir.
- [ ] Las explicaciones de los ejercicios 5 y 6 hablan del mecanismo, no del resultado.
- [ ] **Kernel → Restart & Run All**, y el cuaderno corre entero de arriba abajo.
- [ ] `git add`, `git commit -m "Laboratorio 2"`, `git push`.
- [ ] Las dos líneas pegadas en Moodle.
